# Zero-Trust Privacy-Aware Agent Runtime: Multimodal Memory Gate
**Authors:** Yilin Pan & Tyler Hudgins | **Date:** April 2026

This notebook implements a strictly local, "zero-trust" privacy gate for a multimodal RAG agent. It intercepts text and images before they are embedded into the agent's memory (GCP Vector DB) to ensure sensitive personal information (SPI) is safely redacted.

**Key Architecture Features:**
1. **Zero Cloud APIs:** All inference, including text PII detection and image generation, runs locally.
2. **Ephemeral OCR:** OCR is used *only* to locate the bounding boxes of sensitive text in an image. The OCR strings are immediately deleted and never embedded into the vector database.
3. **Black-Box Masking:** Sensitive regions detected in an image are painted solid black at full original resolution, preserving all non-sensitive areas without any resampling or quality loss.

In [ ]:
import sys, os

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        token = os.getenv("GITHUB_TOKEN", "")
    
    if token:
        !git clone https://{token}@github.com/Athelas011/pii-project.git
    else:
        raise RuntimeError("GITHUB_TOKEN not found. Add it in Colab Secrets (key icon in left sidebar).")
    
    %cd pii-project
    !pip install -q -r requirements.txt
    sys.path.insert(0, os.path.abspath("."))

In [ ]:
%pip install easyocr transformers diffusers torch torchvision matplotlib pillow accelerate pdf2image pyyaml

In [ ]:
import os, sys
import matplotlib.pyplot as plt

_nb_dir = os.path.abspath("")
if _nb_dir not in sys.path:
    sys.path.insert(0, _nb_dir)

from src.privacy_pipeline import (
    initialize_models, setup_output_dir,
    detect_privacy_risks, privacy_gate_and_embed, redact_text,
    load_images_from_path, visualize_detection_result,
)
from src.privacy.box_utils import draw_boxes_on_image

OUTPUTS = setup_output_dir()
print(f"Pipeline imported. Outputs → {OUTPUTS}")

## Phase 0: Model Initialization

All inference runs locally — no data leaves the machine.

| Model | Role |
|---|---|
| `openai/privacy-filter` | NER-based text PII token classification |
| `EasyOCR` | Locate text bounding boxes in images (strings discarded immediately) |
| `google/owlvit-base-patch32` | Zero-shot visual object detection |
| `openai/clip-vit-base-patch32` | Safe CLIP embedding after gate |

All pipeline logic lives in `src/privacy_pipeline.py` — this notebook imports and calls it.

In [ ]:
initialize_models()
print("Ready.")

## Phase 1 & 2: Sensing Gate — Privacy Detection

All detection functions are in `src/privacy_pipeline.py`:

- **`detect_privacy_risks(user_text, image_path, use_ocr)`** — orchestrates text + image detection,
  returning per-page dicts with `redacted_text` and `sensitive_boxes`.
- **`detect_privacy_risks_from_image(image, use_ocr)`** — OwlViT + ephemeral OCR for one image.
- **`redact_text(text)`** — replaces PII spans with `[ENTITY_TYPE]` tags.

OCR strings are deleted immediately after use (ephemeral OCR principle).

## Phase 3 & 4: Policy Engine and Memory Write Gate

All gate logic is in `src/privacy_pipeline.py`:

- **`privacy_gate_and_embed(image, redacted_text, sensitive_boxes)`** — stage-aware policy + CLIP embedding.
  - **ALLOW** — no sensitive regions detected; raw image embedded.
  - **MASK** (<30% coverage) — detected boxes painted solid black at original resolution; masked image embedded.
  - **ABSTRACT** (≥30% coverage) — image blocked; text summary embedded instead.

## Phase 5: Execution & Visualization

Run test payloads through the multimodal gate to verify text redaction,
image inpainting / abstraction, and safe CLIP embedding generation.

Three demo scenarios:
1. **Text + Image** — combined multimodal input
2. **Text only** — batch of PII-laden prompts, no image
3. **Image / PDF only** — visual-only inputs with and without OCR

In [ ]:
test_image_path  = "data/test_upload.jpg"
test_user_prompt = (
    "Save this to my memory: my new phone number is 555-0199 "
    "and here is a photo of my desk."
)

print("=" * 62)
print("  DEMO 1 — Text + Image")
print("=" * 62)

results = detect_privacy_risks(test_user_prompt, test_image_path)
r = results[0]

policy, final_img, summary, embeddings, safe_text = privacy_gate_and_embed(
    image=r["image"],
    redacted_text=r["redacted_text"],
    sensitive_boxes=r["sensitive_boxes"],
)

print(f"\n  Policy enforced : {policy}")
print(f"  Original text   : {test_user_prompt}")
print(f"  Redacted text   : {safe_text}")
print(f"  System message  : {summary}")
print(f"  Sensitive boxes : {len(r['sensitive_boxes'])} region(s) detected")

out_dir = OUTPUTS / "demo1_text_image"
out_dir.mkdir(exist_ok=True)
fig = visualize_detection_result(
    r, policy=policy, final_image=final_img,
    title="Demo 1 — Text + Image",
    save_path=out_dir / "scenario_figure.png",
)
plt.show()

print("\n  Embedding summary:")
if embeddings["image_embeds"] is not None:
    print(f"    Image : {tuple(embeddings['image_embeds'].shape)}")
if embeddings["text_embeds"] is not None:
    print(f"    Text  : {tuple(embeddings['text_embeds'].shape)}")

In [ ]:
test_prompts = [
    "Save this: I used my credit card to buy concert tickets. Card # 123-456-7890, CVV 215.",
    "My SSN is 102-93-8564.",
    "My ex-wife lives at 4000 Mayflower Hill Drive.",
]

print("=" * 62)
print("  DEMO 2 — Text Only (batch)")
print("=" * 62)

for i, prompt in enumerate(test_prompts, 1):
    print(f"\n  ── Test {i} {'─' * 48}")

    results = detect_privacy_risks(user_text=prompt, image_path=None)
    r = results[0]

    policy, _final_img, summary, embeddings, safe_text = privacy_gate_and_embed(
        image=r["image"],
        redacted_text=r["redacted_text"],
        sensitive_boxes=r["sensitive_boxes"],
    )

    print(f"  Policy   : {policy}")
    print(f"  Original : {prompt}")
    print(f"  Redacted : {safe_text}")
    if embeddings["text_embeds"] is not None:
        print(f"  Embedding: {tuple(embeddings['text_embeds'].shape)}")

In [ ]:
test_image_paths = [
    "data/test_upload.jpg",
    "data/test_codebase.png",
    "data/test_email.png",
    "data/test_prescription.pdf",
]

print("=" * 62)
print("  DEMO 3 — Image / PDF Only")
print("=" * 62)

out_dir = OUTPUTS / "demo3_image_pdf"
out_dir.mkdir(exist_ok=True)

for file_i, path in enumerate(test_image_paths, 1):
    fname = os.path.basename(path)
    base  = os.path.splitext(fname)[0]
    print(f"\n\n  ── File {file_i}: {fname} {'─' * max(0, 44 - len(fname))}")

    results = detect_privacy_risks(user_text=None, image_path=path)

    for r in results:
        page_idx  = r["page_index"]
        page_lbl  = f"_p{page_idx + 1}" if page_idx is not None else ""
        page_disp = f" (page {page_idx + 1})" if page_idx is not None else ""

        print(f"\n  Processing{page_disp}…")

        policy, final_img, summary, embeddings, _ = privacy_gate_and_embed(
            image=r["image"],
            redacted_text=r["redacted_text"],
            sensitive_boxes=r["sensitive_boxes"],
        )

        print(f"  Policy          : {policy}")
        print(f"  Sensitive boxes : {len(r['sensitive_boxes'])}")
        print(f"  System message  : {summary}")

        if r["image"] is None:
            print("  [ No valid image found ]")
            continue

        fig = visualize_detection_result(
            r, policy=policy, final_image=final_img,
            title=f"{fname}{page_disp}",
            save_path=out_dir / f"{base}{page_lbl}_figure.png",
        )
        plt.show()

        if embeddings["image_embeds"] is not None:
            print(f"  Image embedding : {tuple(embeddings['image_embeds'].shape)}")
        if embeddings["text_embeds"] is not None:
            print(f"  Text  embedding : {tuple(embeddings['text_embeds'].shape)}")